# Chapter 7: Cleaning Structural Noise: Duplicates, Outliers, and Inconsistencies

Companion notebook for *Master Data Cleaning: A Hands-On Beginner's Guide with Python, pandas & NumPy*.

Run the cells from top to bottom (**Kernel → Restart Kernel and Run All Cells**). The data files this chapter uses are in the same folder as this notebook.

## 7.1 Identifying Duplicate Rows

### `df.duplicated()`

In [1]:
import numpy as np
import pandas as pd

customers = pd.DataFrame({
    'customer_id': [101, 102, 101, 103, 102, 104],
    'name': ['Alice', 'Bob', 'Alice', 'Charlie', 'Bob', 'Diana'],
    'purchase': [250, 150, 250, 300, 150, 200],
})
customers

,customer_id,name,purchase
0,101,Alice,250
1,102,Bob,150
2,101,Alice,250
3,103,Charlie,300
4,102,Bob,150
5,104,Diana,200


In [2]:
customers.duplicated()

0    False
1    False
2     True
3    False
4     True
5    False
dtype: bool

In [3]:
customers[customers.duplicated()]

,customer_id,name,purchase
2,101,Alice,250
4,102,Bob,150


### Choosing which copy is flagged with `keep`

In [4]:
pd.DataFrame({
    'first': customers.duplicated(keep='first'),
    'last': customers.duplicated(keep='last'),
    'all': customers.duplicated(keep=False),
})

,first,last,all
0,False,True,True
1,False,True,True
2,True,False,True
3,False,False,False
4,True,False,True
5,False,False,False


In [5]:
customers[customers.duplicated(keep=False)].sort_values('customer_id')

,customer_id,name,purchase
0,101,Alice,250
2,101,Alice,250
1,102,Bob,150
4,102,Bob,150


### Checking only some columns with `subset`

In [6]:
transactions = pd.DataFrame({
    'transaction_id': [1001, 1002, 1003, 1004, 1005],
    'customer_id': [101, 102, 101, 103, 102],
    'amount': [250, 150, 275, 300, 150],
    'date': ['2024-01-03', '2024-01-02', '2024-01-01',
             '2024-01-04', '2024-01-05'],
})
transactions

,transaction_id,customer_id,amount,date
0,1001,101,250,2024-01-03
1,1002,102,150,2024-01-02
2,1003,101,275,2024-01-01
3,1004,103,300,2024-01-04
4,1005,102,150,2024-01-05


In [7]:
transactions.duplicated(subset=['customer_id'])

0    False
1    False
2     True
3    False
4     True
dtype: bool

In [8]:
transactions.duplicated(subset=['customer_id', 'amount'])

0    False
1    False
2    False
3    False
4     True
dtype: bool

### Counting duplicates

In [9]:
n_dupes = customers.duplicated().sum()
pct_dupes = n_dupes / len(customers) * 100
print(f'Duplicates: {n_dupes} ({pct_dupes:.1f}% of rows)')

Duplicates: 2 (33.3% of rows)


In [10]:
names = pd.Series(['Alice Smith', 'Alice Smith ', 'alice smith'])
names.duplicated()

0    False
1    False
2    False
dtype: bool

In [11]:
names.str.strip().str.lower().duplicated()

0    False
1     True
2     True
dtype: bool

## 7.2 Removing Duplicates

### `df.drop_duplicates()`

In [12]:
customers.drop_duplicates()

,customer_id,name,purchase
0,101,Alice,250
1,102,Bob,150
3,103,Charlie,300
5,104,Diana,200


In [13]:
customers_clean = customers.drop_duplicates().reset_index(drop=True)
customers_clean

,customer_id,name,purchase
0,101,Alice,250
1,102,Bob,150
2,103,Charlie,300
3,104,Diana,200


In [14]:
customers.drop_duplicates(keep='last')

,customer_id,name,purchase
2,101,Alice,250
3,103,Charlie,300
4,102,Bob,150
5,104,Diana,200


In [15]:
customers.drop_duplicates(keep=False)

,customer_id,name,purchase
3,103,Charlie,300
5,104,Diana,200


### One row per key with `subset`

In [16]:
transactions.drop_duplicates(subset=['customer_id'])

,transaction_id,customer_id,amount,date
0,1001,101,250,2024-01-03
1,1002,102,150,2024-01-02
3,1004,103,300,2024-01-04


In [17]:
transactions.drop_duplicates(subset=['customer_id', 'amount'])

,transaction_id,customer_id,amount,date
0,1001,101,250,2024-01-03
1,1002,102,150,2024-01-02
2,1003,101,275,2024-01-01
3,1004,103,300,2024-01-04


### Sorting before `keep='last'`

In [18]:
transactions.drop_duplicates(subset=['customer_id'], keep='last')

,transaction_id,customer_id,amount,date
2,1003,101,275,2024-01-01
3,1004,103,300,2024-01-04
4,1005,102,150,2024-01-05


In [19]:
latest = (
    transactions.sort_values('date')
    .drop_duplicates(subset=['customer_id'], keep='last')
)
latest

,transaction_id,customer_id,amount,date
0,1001,101,250,2024-01-03
3,1004,103,300,2024-01-04
4,1005,102,150,2024-01-05


### Putting it together

In [20]:
signups = pd.read_csv('signups.csv')
signups['email'] = signups['email'].str.strip().str.lower()
n_repeats = signups.duplicated(subset=['email']).sum()
print(f'Repeated emails: {n_repeats}')

Repeated emails: 4


In [21]:
dupes = signups[signups.duplicated(subset=['email'], keep=False)]
dupes.sort_values(['email', 'signup_date'])

,email,name,plan,signup_date
0,aisha@example.com,Aisha Rahman,Free,2024-03-02
2,aisha@example.com,Aisha Rahman,Pro,2024-04-18
1,ben.lee@example.com,Ben Lee,Free,2024-03-05
4,ben.lee@example.com,Ben Lee,Free,2024-03-05
7,chen.wei@example.com,Chen Wei,Team,2024-02-20
3,chen.wei@example.com,Chen Wei,Pro,2024-03-09
8,dina@example.com,Dina Osman,Free,2024-03-12
5,dina@example.com,Dina Osman,Pro,2024-05-01


In [22]:
rows_before = len(signups)
signups = (
    signups.sort_values('signup_date')
    .drop_duplicates(subset=['email'], keep='last')
    .reset_index(drop=True)
)
print(f'Removed {rows_before - len(signups)} rows, '
      f'{len(signups)} customers left')
signups

Removed 4 rows, 5 customers left


,email,name,plan,signup_date
0,ben.lee@example.com,Ben Lee,Free,2024-03-05
1,chen.wei@example.com,Chen Wei,Pro,2024-03-09
2,farid@example.com,Farid Hakim,Free,2024-03-15
3,aisha@example.com,Aisha Rahman,Pro,2024-04-18
4,dina@example.com,Dina Osman,Pro,2024-05-01


## 7.3 Data Validation: Using Unique Values and Set Logic

In [23]:
orders = pd.DataFrame({
    'order_id': [101, 102, 103, 104, 105, 106, 107, 108],
    'customer': ['Alice', 'Bob', 'Charlie', 'Diana',
                 'Eve', 'Frank', 'Grace', 'Henry'],
    'status': ['Complete', 'Pending ', 'failed', 'Complete',
               'Canceled', 'Complete', 'pending', 'Failed'],
    'amount': [250.0, 150.0, 75.5, 300.0, 125.0, 200.0, 180.0, 90.0],
})
orders

,order_id,customer,status,amount
0,101,Alice,Complete,250.0
1,102,Bob,Pending,150.0
2,103,Charlie,failed,75.5
3,104,Diana,Complete,300.0
4,105,Eve,Canceled,125.0
5,106,Frank,Complete,200.0
6,107,Grace,pending,180.0
7,108,Henry,Failed,90.0


### `s.unique()`

In [24]:
orders['status'].unique()

<ArrowStringArray>
['Complete', 'Pending ', 'failed', 'Canceled', 'pending', 'Failed']
Length: 6, dtype: str

### `np.unique()`

In [25]:
np.unique(orders['status'])

array(['Canceled', 'Complete', 'Failed', 'Pending ', 'failed', 'pending'],
      dtype=object)

In [26]:
values, counts = np.unique(orders['status'], return_counts=True)
print(values)
print(counts)

['Canceled' 'Complete' 'Failed' 'Pending ' 'failed' 'pending']
[1 3 1 1 1 1]


### `s.nunique()`

In [27]:
orders['status'].nunique()

6

In [28]:
orders_missing = orders.copy()
orders_missing.loc[8] = [109, 'Iris', np.nan, 220.0]
print('Excluding NaN:', orders_missing['status'].nunique())
print('Including NaN:', orders_missing['status'].nunique(dropna=False))

Excluding NaN: 6
Including NaN: 7


### `s.value_counts()`

In [29]:
orders['status'].value_counts()

status
Complete    3
Pending     1
failed      1
Canceled    1
pending     1
Failed      1
Name: count, dtype: int64

In [30]:
orders['status'].value_counts(normalize=True)

status
Complete    0.375
Pending     0.125
failed      0.125
Canceled    0.125
pending     0.125
Failed      0.125
Name: proportion, dtype: float64

In [31]:
orders_missing['status'].value_counts(dropna=False)

status
Complete    3
Pending     1
failed      1
Canceled    1
pending     1
Failed      1
NaN         1
Name: count, dtype: int64

### Comparing against the allowed values with sets

In [32]:
allowed = {'Complete', 'Pending', 'Failed', 'Canceled'}
found = set(orders['status'].unique())
sorted(found - allowed)

['Pending ', 'failed', 'pending']

In [33]:
sorted(allowed - found)

['Pending']

In [34]:
orders[~orders['status'].isin(allowed)]

,order_id,customer,status,amount
1,102,Bob,Pending,150.0
2,103,Charlie,failed,75.5
6,107,Grace,pending,180.0


In [35]:
fixed = orders['status'].str.strip().str.title()
set(fixed.unique()).issubset(allowed)

True

## 7.4 Detecting Outliers: Filtering by Standard Deviation or Quantile Range

In [36]:
rents = pd.DataFrame({
    'apartment_id': ['A01', 'A02', 'A03', 'A04', 'A05', 'A06',
                     'A07', 'A08', 'A09', 'A10', 'A11', 'A12'],
    'monthly_rent': [1200, 1350, 1100, 1250, 8500, 1400,
                     1150, 1300, 1275, 250, 1225, 1180],
    'square_feet': [650, 720, 600, 680, 700, 750,
                    620, 690, 670, 640, 660, 640],
})
rents

,apartment_id,monthly_rent,square_feet
0,A01,1200,650
1,A02,1350,720
2,A03,1100,600
3,A04,1250,680
4,A05,8500,700
5,A06,1400,750
6,A07,1150,620
7,A08,1300,690
8,A09,1275,670
9,A10,250,640


### `s.quantile()`

In [37]:
rent = rents['monthly_rent']
q1 = rent.quantile(0.25)
q3 = rent.quantile(0.75)
iqr = q3 - q1
print(f'Q1: {q1}, median: {rent.median()}, Q3: {q3}, IQR: {iqr}')

Q1: 1172.5, median: 1237.5, Q3: 1312.5, IQR: 140.0


### The IQR rule

In [38]:
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
print(f'Lower fence: {lower}, upper fence: {upper}')

Lower fence: 962.5, upper fence: 1522.5


In [39]:
rents[(rent < lower) | (rent > upper)]

,apartment_id,monthly_rent,square_feet
4,A05,8500,700
9,A10,250,640


In [40]:
is_outlier = ~rent.between(lower, upper)
rents[is_outlier]

,apartment_id,monthly_rent,square_feet
4,A05,8500,700
9,A10,250,640


### Z-scores with `s.mean()` and `s.std()`

In [41]:
mean_rent = rent.mean()
std_rent = rent.std()
rents['z_score'] = ((rent - mean_rent) / std_rent).round(2)
print(f'Mean: {mean_rent}, std: {std_rent:.2f}')
rents

Mean: 1765.0, std: 2141.71


,apartment_id,monthly_rent,square_feet,z_score
0,A01,1200,650,-0.26
1,A02,1350,720,-0.19
2,A03,1100,600,-0.31
3,A04,1250,680,-0.24
4,A05,8500,700,3.14
5,A06,1400,750,-0.17
6,A07,1150,620,-0.29
7,A08,1300,690,-0.22
8,A09,1275,670,-0.23
9,A10,250,640,-0.71


In [42]:
rents[rents['z_score'].abs() > 3]

,apartment_id,monthly_rent,square_feet,z_score
4,A05,8500,700,3.14


In [43]:
rents = rents.drop(columns='z_score')

### Sample vs. population standard deviation

In [44]:
print(f'pandas std():        {rent.std():.2f}')
print(f'np.std():            {np.std(rent):.2f}')
print(f'pandas std(ddof=0):  {rent.std(ddof=0):.2f}')

pandas std():        2141.71
np.std():            2050.53
pandas std(ddof=0):  2050.53


### Percentile cutoffs

In [45]:
cutoffs = rent.quantile([0.05, 0.95])
cutoffs

0.05     717.5
0.95    4595.0
Name: monthly_rent, dtype: float64

In [46]:
rents[~rent.between(cutoffs[0.05], cutoffs[0.95])]

,apartment_id,monthly_rent,square_feet
4,A05,8500,700
9,A10,250,640


### Choosing a method

## 7.5 Addressing Outliers: Winsorizing, Replacement, or Removal

### Flagging outliers

In [47]:
rents_flagged = rents.copy()
rents_flagged['rent_outlier'] = ~rents['monthly_rent'].between(
    lower, upper)
rents_flagged

,apartment_id,monthly_rent,square_feet,rent_outlier
0,A01,1200,650,False
1,A02,1350,720,False
2,A03,1100,600,False
3,A04,1250,680,False
4,A05,8500,700,True
5,A06,1400,750,False
6,A07,1150,620,False
7,A08,1300,690,False
8,A09,1275,670,False
9,A10,250,640,True


### Removing outliers

In [48]:
rents_removed = rents[rents['monthly_rent'].between(lower, upper)]
print(f'Rows left: {len(rents_removed)}, '
      f'removed: {len(rents) - len(rents_removed)}')

Rows left: 10, removed: 2


In [49]:
before = rents['monthly_rent']
after = rents_removed['monthly_rent']
print(f'Mean:   {before.mean():.2f} -> {after.mean():.2f}')
print(f'Median: {before.median():.2f} -> {after.median():.2f}')
print(f'Std:    {before.std():.2f} -> {after.std():.2f}')

Mean:   1765.00 -> 1243.00
Median: 1237.50 -> 1237.50
Std:    2141.71 -> 91.69


### Capping with `s.clip()`

In [50]:
rents_capped = rents.copy()
rents_capped['monthly_rent'] = rents['monthly_rent'].clip(lower, upper)
rents_capped

,apartment_id,monthly_rent,square_feet
0,A01,1200.0,650
1,A02,1350.0,720
2,A03,1100.0,600
3,A04,1250.0,680
4,A05,1522.5,700
5,A06,1400.0,750
6,A07,1150.0,620
7,A08,1300.0,690
8,A09,1275.0,670
9,A10,962.5,640


In [51]:
rents['monthly_rent'].clip(cutoffs[0.05], cutoffs[0.95])

0     1200.0
1     1350.0
2     1100.0
3     1250.0
4     4595.0
5     1400.0
6     1150.0
7     1300.0
8     1275.0
9      717.5
10    1225.0
11    1180.0
Name: monthly_rent, dtype: float64

### Replacing outliers with `np.where()`

In [52]:
median_rent = rents['monthly_rent'].median()
outlier_mask = ~rents['monthly_rent'].between(lower, upper)

rents_replaced = rents.copy()
rents_replaced['monthly_rent'] = np.where(
    outlier_mask, median_rent, rents['monthly_rent'])
rents_replaced

,apartment_id,monthly_rent,square_feet
0,A01,1200.0,650
1,A02,1350.0,720
2,A03,1100.0,600
3,A04,1250.0,680
4,A05,1237.5,700
5,A06,1400.0,750
6,A07,1150.0,620
7,A08,1300.0,690
8,A09,1275.0,670
9,A10,1237.5,640


### Comparing the results

In [53]:
treatments = {
    'original': rents['monthly_rent'],
    'removed': rents_removed['monthly_rent'],
    'capped': rents_capped['monthly_rent'],
    'replaced': rents_replaced['monthly_rent'],
}
pd.DataFrame({
    'mean': {k: v.mean() for k, v in treatments.items()},
    'std': {k: v.std() for k, v in treatments.items()},
    'rows': {k: len(v) for k, v in treatments.items()},
}).round(2)

,mean,std,rows
original,1765.00,2141.71,12
removed,1243.00,91.69,10
capped,1242.92,145.37,12
replaced,1242.08,82.96,12
